# 10 · Final report: test scores, significance, breakdowns, errors

**Goal:** score the chosen configurations on **test** (saved blind since Direction 1), compare them with paired tests, and produce the tables of the written report. No configuration is chosen here: every choice was made on val (B, D1, D2) or on train CV + val (D3). A new run (e.g. a later cross-encoder) only needs one line in `RUNS`.

**Two versions of Direction 3.** The first LightGBM (base CE only) was the final system until DeBERTa-large was run: its test score, and those of the single large model and of their blend, were seen before the large model was added to 08 / 09. The second version (base + large CE) was chosen by the rule written in 09 before its run (CV and val only); its test scores, and those of the 5-fold large CE, are scored here for the first time.

**Additional experiments** (stage `Extra`, run after the first LightGBM was chosen, not used to choose): DeBERTa ce4 trained for 2 epochs, the bge-reranker-base cross-encoder (`11_bge_reranker`), DeBERTa-v3-large (`12_deberta_large_t4x2`), and score blends: scores standardized within each query, then weighted. Each blend weight is the best of 0.1–0.5 on val (bge 0.2 with DeBERTa-base, used again for the MiniLM blend and the LightGBM + bge blend; base 0.3 and bge 0.2 with large; large 0.5 with the first LightGBM), so the val numbers of blends are slightly optimistic and test is the fair comparison.

**Sections:** 1 data · 2 main table (val and test, every run) · 3 headline table (mean over seeds) · 4 paired tests · 5 by query type · 6 by query property (length, numbers and units, color, negation, brand) · 7 where each label lands · 8 per-query wins and losses · 9 worst test queries of the final model · 10 supplement: training targets and distillation (val only) · 11 save.

**Kaggle settings:** CPU, no internet, a few minutes. Also runs locally from `notebooks/` (inputs read from `../../kaggle_outputs`).

**Inputs** (Add Input):
- Dataset `wm-src`, a version with the `scores/` of every run below. Required (locally: `../kaggle_outputs/results/scores/`).
- Notebook Output **01_preprocess** (`pairs.parquet`, `products_clean.parquet`). Required.
- Notebook Output **07_oof** (assemble version) or the dataset with `ce_deberta_main_ce4.parquet`, and the assemble version of **12** (`ce_debertalarge_main_ce4.parquet`): test scores of the 5-fold CEs. Required.
- Notebook Outputs **14_distill** (`scores/d4_minilm_main_*_val.parquet`) and **15_deberta_base_s43** (`scores/d2_deberta_main_ce4_s43_val.parquet`). Optional: the rows of section 10 that need them are skipped without them.

**Output** (`/kaggle/working/report/`, locally `data/report/`): `results_all.csv`, `results_headline.csv`, `paired_tests.csv`, `by_source.csv`, `by_query_property.csv`, `label_ranks.csv`, `query_deltas.csv`, `worst_queries.csv`, `supplement_val.csv` (if section 10 ran).

In [1]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

src from D:\Workspace\Web Mining\web-mining-project | {'files_sha1': '62eb710944'}


In [2]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from src.data import PREPROCESS_HINT, find
from src.evaluation import paired_bootstrap, summarize

ON_KAGGLE = Path('/kaggle/input').exists()
INPUT = '/kaggle/input' if ON_KAGGLE else '../../kaggle_outputs'
OUT = Path('/kaggle/working/report') if ON_KAGGLE else ROOT / 'data' / 'report'
SCORES = {Path(p).name: p for p in glob.glob(f'{INPUT}/**/scores/*.parquet', recursive=True)}   # <run>_<split>.parquet
CE5 = 'd3_ce_deberta_main_ce4'          # 5-fold CE: val from its run file, test from the assembled 07 file
CE5L = 'd3_ce_debertalarge_main_ce4'    # same for DeBERTa-large (test from the assembled 12 file)
FIRST = 'd3_lgbm_multiclass'            # first LightGBM: DeBERTa-base CE only (chosen on CV + val)
BLEND = 'd3_blend_lgbm_large'           # 0.4 FIRST + 0.6 large CE (5 folds), weight chosen on CV in 09
FINAL = 'd3_lgbm_ce3_lambdarank'        # final system: LightGBM on both CEs (rule in 09, CV + val)

# (stage, configuration, seed, run name); order = order of the tables
RUNS = [
    ('B0', 'Random order', None, 'b0_random'),
    ('B1', 'BM25 title', None, 'b1_bm25_title'),
    ('B1', 'BM25 main', None, 'b1_bm25_main'),
    ('B2', 'Bi-encoder MiniLM-L6 title (zero-shot)', None, 'b2_minilml6_title'),
    ('B2', 'Bi-encoder MiniLM-L6 main (zero-shot)', None, 'b2_minilml6_main'),
    ('B2', 'Bi-encoder bge-base title (zero-shot)', None, 'b2_bge_title'),
    ('B2', 'Bi-encoder bge-base main (zero-shot)', None, 'b2_bge_main'),
    ('B3', 'CE MiniLM MS MARCO title (zero-shot)', None, 'b3_minilm_title_zeroshot'),
    ('B3', 'CE MiniLM title, MSE', 42, 'b3_minilm_title_mse_s42'),
    ('B3', 'CE MiniLM title, MSE', 43, 'b3_minilm_title_mse_s43'),
    ('D1', 'CE MiniLM title_bc, MSE', 42, 'd1_minilm_title_bc_mse_s42'),
    ('D1', 'CE MiniLM title_bc, MSE', 43, 'd1_minilm_title_bc_mse_s43'),
    ('D1', 'CE MiniLM main, MSE', 42, 'd1_minilm_main_mse_s42'),
    ('D1', 'CE MiniLM main, MSE', 43, 'd1_minilm_main_mse_s43'),
    ('D1', 'CE DeBERTa title, MSE', 42, 'd1_deberta_title_mse_s42'),
    ('D1', 'CE DeBERTa main, MSE', 42, 'd1_deberta_main_mse_s42'),
    ('D1', 'CE DeBERTa main, MSE', 43, 'd1_deberta_main_mse_s43'),
    ('D2', 'CE MiniLM title, ce4', 42, 'd2_minilm_title_ce4_s42'),
    ('D2', 'CE MiniLM title, ce4', 43, 'd2_minilm_title_ce4_s43'),
    ('D2', 'CE MiniLM title, ListNet', 42, 'd2_minilm_title_listnet_s42'),
    ('D2', 'CE MiniLM title, ListNet', 43, 'd2_minilm_title_listnet_s43'),
    ('D2', 'CE DeBERTa main, ce4', 42, 'd2_deberta_main_ce4_s42'),
    ('D3', 'CE DeBERTa main ce4, mean of 5 fold models', 42, CE5),
    ('D3', 'LightGBM lambdarank on base CE + features', 42, 'd3_lgbm_lambdarank'),
    ('D3', 'LightGBM multiclass on base CE + features', 42, 'd3_lgbm_multiclass'),
    ('D3', 'CE DeBERTa-large main ce4, mean of 5 fold models', 42, CE5L),
    ('D3', 'Blend: LightGBM (base CE) + 0.6 large CE (5 folds)', 42, BLEND),
    ('D3', 'LightGBM multiclass on base + large CE + features', 42, 'd3_lgbm_ce3_multiclass'),
    ('D3', 'LightGBM lambdarank on base + large CE + features', 42, FINAL),
    ('Extra', 'CE DeBERTa main, ce4, 2 epochs', 42, 'd2_deberta_main_e2_ce4_s42'),
    ('Extra', 'CE bge-reranker-base main, ce4', 42, 'd1_bgerr_main_ce4_s42'),
    ('Extra', 'Blend: DeBERTa ce4 + 0.2 bge-reranker', 42, 'x_deberta_bgerr'),
    ('Extra', 'Blend: DeBERTa ce4 + 0.2 MiniLM main', 42, 'x_deberta_minilm'),
    ('Extra', 'Blend: LightGBM (base CE) + 0.2 bge-reranker', 42, 'x_final_bgerr'),
    ('Extra', 'CE DeBERTa-v3-large main, ce4', 42, 'd1_debertalarge_main_ce4_s42'),
    ('Extra', 'Blend: large + 0.3 DeBERTa-base', 42, 'x_large_base'),
    ('Extra', 'Blend: large + 0.2 bge-reranker', 42, 'x_large_bgerr'),
    ('Extra', 'Blend: LightGBM (base CE) + 0.5 large (1 model)', 42, 'x_final_large'),
]

# blends of runs listed above: within-query z-scores, weighted sum (see the introduction)
BLENDS = {
    'x_deberta_bgerr': {'d2_deberta_main_ce4_s42': 0.8, 'd1_bgerr_main_ce4_s42': 0.2},
    'x_deberta_minilm': {'d2_deberta_main_ce4_s42': 0.8, 'd1_minilm_main_mse_s42': 0.2},
    'x_final_bgerr': {FIRST: 0.8, 'd1_bgerr_main_ce4_s42': 0.2},
    'x_large_base': {'d1_debertalarge_main_ce4_s42': 0.7, 'd2_deberta_main_ce4_s42': 0.3},
    'x_large_bgerr': {'d1_debertalarge_main_ce4_s42': 0.8, 'd1_bgerr_main_ce4_s42': 0.2},
    'x_final_large': {FIRST: 0.5, 'd1_debertalarge_main_ce4_s42': 0.5},
}

# (question, run a, run b): paired difference b - a on the same queries
PAIRS = [
    ('B2 bge main vs BM25 main', 'b1_bm25_main', 'b2_bge_main'),
    ('B3 fine-tuning vs zero-shot CE', 'b3_minilm_title_zeroshot', 'b3_minilm_title_mse_s42'),
    ('B3 vs BM25 main', 'b1_bm25_main', 'b3_minilm_title_mse_s42'),
    ('D1 MiniLM: title_bc vs title (s42)', 'b3_minilm_title_mse_s42', 'd1_minilm_title_bc_mse_s42'),
    ('D1 MiniLM: main vs title (s42)', 'b3_minilm_title_mse_s42', 'd1_minilm_main_mse_s42'),
    ('D1 DeBERTa: main vs title', 'd1_deberta_title_mse_s42', 'd1_deberta_main_mse_s42'),
    ('D1 main: DeBERTa vs MiniLM (s42)', 'd1_minilm_main_mse_s42', 'd1_deberta_main_mse_s42'),
    ('D1 main: DeBERTa vs MiniLM (s43)', 'd1_minilm_main_mse_s43', 'd1_deberta_main_mse_s43'),
    ('D1 seed: DeBERTa main s43 vs s42', 'd1_deberta_main_mse_s42', 'd1_deberta_main_mse_s43'),
    ('D2 MiniLM: ce4 vs MSE (s42)', 'b3_minilm_title_mse_s42', 'd2_minilm_title_ce4_s42'),
    ('D2 MiniLM: ce4 vs MSE (s43)', 'b3_minilm_title_mse_s43', 'd2_minilm_title_ce4_s43'),
    ('D2 MiniLM: ListNet vs MSE (s42)', 'b3_minilm_title_mse_s42', 'd2_minilm_title_listnet_s42'),
    ('D2 MiniLM: ListNet vs MSE (s43)', 'b3_minilm_title_mse_s43', 'd2_minilm_title_listnet_s43'),
    ('D2 DeBERTa main: ce4 vs MSE', 'd1_deberta_main_mse_s42', 'd2_deberta_main_ce4_s42'),
    ('D3 mean of 5 fold models vs 1 model on all train', 'd2_deberta_main_ce4_s42', CE5),
    ('D3 LightGBM lambdarank vs CE (5 folds)', CE5, 'd3_lgbm_lambdarank'),
    ('D3 LightGBM multiclass vs CE (5 folds)', CE5, 'd3_lgbm_multiclass'),
    ('D3 multiclass vs lambdarank', 'd3_lgbm_lambdarank', 'd3_lgbm_multiclass'),
    ('D3 large CE (5 folds) vs base CE (5 folds)', CE5, CE5L),
    ('D3 large CE: 5 fold models vs 1 model on all train', 'd1_debertalarge_main_ce4_s42', CE5L),
    ('D3 large CE (5 folds) vs LightGBM (base CE)', FIRST, CE5L),
    ('D3 blend vs large CE (5 folds)', CE5L, BLEND),
    ('D3 final vs large CE (5 folds)', CE5L, FINAL),
    ('D3 final vs blend', BLEND, FINAL),
    ('D3 final vs LightGBM (base CE)', FIRST, FINAL),
    ('D3 final (lambdarank) vs multiclass, base + large CE', 'd3_lgbm_ce3_multiclass', FINAL),
    ('Final vs B3', 'b3_minilm_title_mse_s42', FINAL),
    ('Final vs BM25 main', 'b1_bm25_main', FINAL),
    ('Extra: DeBERTa ce4 2 epochs vs 1 epoch', 'd2_deberta_main_ce4_s42', 'd2_deberta_main_e2_ce4_s42'),
    ('Extra: bge-reranker vs DeBERTa (main, ce4)', 'd2_deberta_main_ce4_s42', 'd1_bgerr_main_ce4_s42'),
    ('Extra: bge-reranker vs MiniLM (main)', 'd1_minilm_main_mse_s42', 'd1_bgerr_main_ce4_s42'),
    ('Extra: DeBERTa + bge-reranker vs DeBERTa', 'd2_deberta_main_ce4_s42', 'x_deberta_bgerr'),
    ('Extra: DeBERTa + MiniLM vs DeBERTa', 'd2_deberta_main_ce4_s42', 'x_deberta_minilm'),
    ('Extra: LightGBM (base CE) + bge-reranker vs LightGBM (base CE)', FIRST, 'x_final_bgerr'),
    ('Extra: DeBERTa large vs base (main, ce4)', 'd2_deberta_main_ce4_s42', 'd1_debertalarge_main_ce4_s42'),
    ('Extra: DeBERTa large vs LightGBM (base CE)', FIRST, 'd1_debertalarge_main_ce4_s42'),
    ('Extra: large + base vs large', 'd1_debertalarge_main_ce4_s42', 'x_large_base'),
    ('Extra: large + bge-reranker vs large', 'd1_debertalarge_main_ce4_s42', 'x_large_bgerr'),
    ('Extra: LightGBM (base CE) + large vs LightGBM (base CE)', FIRST, 'x_final_large'),
]
BREAKDOWN = ['b1_bm25_main', 'b2_bge_main', 'b3_minilm_title_mse_s42', 'd1_deberta_main_mse_s42', CE5, CE5L, FINAL]
NAME = {r: (f'{c} (s{s})' if s and sum(x[1] == c for x in RUNS) > 1 else c) for _, c, s, r in RUNS}

## 1. Data and scores

Labels come from `pairs.parquet`. Every run's scores are aligned to the pairs by `example_id` and must cover every pair of the split.

In [3]:
pairs = pd.read_parquet(find('pairs.parquet', PREPROCESS_HINT, INPUT),
                        columns=['example_id', 'query_id', 'query', 'product_id', 'esci_label', 'source', 'split'])
parts = {s: pairs[pairs['split'] == s].sort_values(['query_id', 'example_id']).reset_index(drop=True) for s in ['val', 'test']}
CE_TEST = {}   # assembled test scores of the 5-fold CEs
for run, f, nb in [(CE5, 'ce_deberta_main_ce4.parquet', '07_oof'), (CE5L, 'ce_debertalarge_main_ce4.parquet', '12_deberta_large_t4x2')]:
    t = pd.read_parquet(find(f, f'Add Input -> {nb} (assemble) output', INPUT))
    CE_TEST[run] = t[t['split'] == 'test'][['example_id', 'score']]


def scores(run, split):
    f = f'{run}_{split}.parquet'
    if f in SCORES:
        sc = pd.read_parquet(SCORES[f], columns=['example_id', 'score'])
    elif run in CE_TEST and split == 'test':
        sc = CE_TEST[run]
    else:
        raise FileNotFoundError(f'{f}: not in a scores/ folder under {INPUT}; rebuild wm-src with the latest scores')
    s = parts[split][['example_id']].merge(sc, on='example_id', how='left')['score']
    assert s.notna().all(), f'{run} {split}: scores do not cover every pair'
    return s.to_numpy()


def zscore(split, s):
    """Scores standardized within each query (0 for one-item or constant lists)."""
    s = pd.Series(s)
    g = s.groupby(parts[split]['query_id'].to_numpy())
    return ((s - g.transform('mean')) / g.transform('std').replace(0, np.nan)).fillna(0).to_numpy()


metrics, per_query, score_cache = {}, {}, {}
for _, _, _, run in RUNS:
    for split in ['val', 'test']:
        if run in BLENDS:
            score_cache[run, split] = sum(w * zscore(split, score_cache[r, split]) for r, w in BLENDS[run].items())
        else:
            score_cache[run, split] = scores(run, split)
        metrics[run, split], per_query[run, split] = summarize(parts[split], score_cache[run, split])
print(len(RUNS), 'runs scored on val and test |', {s: f"{p['query_id'].nunique():,} queries" for s, p in parts.items()})

38 runs scored on val and test | {'val': '2,089 queries', 'test': '8,956 queries'}


## 2. Every run (val and test)

nDCG with official gains (E 1, S 0.1, C 0.01, I 0), ties broken at random (3 seeds), 95% bootstrap CI over queries. `ndcg_amazon` uses the gains of Amazon's script (S and C swapped), for comparison with the paper only.

In [4]:
rows = []
for stage, config, seed, run in RUNS:
    v, t = metrics[run, 'val'], metrics[run, 'test']
    rows.append({'stage': stage, 'configuration': config, 'seed': seed, 'run': run,
                 'val_ndcg': v['ndcg'], 'test_ndcg': t['ndcg'], 'test_ci95': t['ndcg_ci95'],
                 'test_ndcg_amazon': t['ndcg_amazon'], 'test_ndcg10': t['ndcg10'], 'test_mrr_e': t['mrr_e'],
                 'test_e_over_s': t['e_over_s']})
results = pd.DataFrame(rows)
pd.set_option('display.width', 250, 'display.max_columns', 30, 'display.max_colwidth', 60)
display(results.drop(columns='run'))

,stage,configuration,seed,val_ndcg,test_ndcg,test_ci95,test_ndcg_amazon,test_ndcg10,test_mrr_e,test_e_over_s
0,B0,Random order,NaN,0.7422,0.7459,"[0.7429, 0.7491]",0.7133,0.5763,0.6087,0.5000
1,B1,BM25 title,NaN,0.8162,0.8209,"[0.8179, 0.824]",0.7996,0.6954,0.7579,0.6103
2,B1,BM25 main,NaN,0.8227,0.8244,"[0.8209, 0.8275]",0.8046,0.7012,0.7595,0.6190
3,B2,Bi-encoder MiniLM-L6 title (zero-shot),NaN,0.8202,0.8281,"[0.8248, 0.8311]",0.8051,0.7091,0.7672,0.6062
4,B2,Bi-encoder MiniLM-L6 main (zero-shot),NaN,0.8264,0.8306,"[0.8276, 0.8339]",0.8081,0.7137,0.7702,0.6136
5,B2,Bi-encoder bge-base title (zero-shot),NaN,0.8418,0.8450,"[0.842, 0.848]",0.8243,0.7366,0.7981,0.6397
6,B2,Bi-encoder bge-base main (zero-shot),NaN,0.8429,0.8471,"[0.844, 0.85]",0.8274,0.7401,0.8023,0.6433
7,B3,CE MiniLM MS MARCO title (zero-shot),NaN,0.8330,0.8340,"[0.8311, 0.8369]",0.8132,0.7163,0.7895,0.6248
8,B3,"CE MiniLM title, MSE",42.0,0.8566,0.8549,"[0.8518, 0.8578]",0.8358,0.7539,0.8174,0.6760
9,B3,"CE MiniLM title, MSE",43.0,0.8578,0.8555,"[0.8523, 0.8586]",0.8367,0.7546,0.8176,0.6786


## 3. Headline table (mean over seeds)

One row per configuration; configurations run with two seeds show the mean and the seed range on test.

In [5]:
headline = (results.groupby(['stage', 'configuration'], sort=False)
            .agg(seeds=('seed', lambda s: ', '.join(str(int(x)) for x in s.dropna()) or '-'),
                 val_ndcg=('val_ndcg', 'mean'), test_ndcg=('test_ndcg', 'mean'),
                 test_min=('test_ndcg', 'min'), test_max=('test_ndcg', 'max'),
                 test_ndcg10=('test_ndcg10', 'mean'), test_mrr_e=('test_mrr_e', 'mean'),
                 test_e_over_s=('test_e_over_s', 'mean'))
            .round(4).reset_index())
display(headline)

,stage,configuration,seeds,val_ndcg,test_ndcg,test_min,test_max,test_ndcg10,test_mrr_e,test_e_over_s
0,B0,Random order,-,0.7422,0.7459,0.7459,0.7459,0.5763,0.6087,0.5000
1,B1,BM25 title,-,0.8162,0.8209,0.8209,0.8209,0.6954,0.7579,0.6103
2,B1,BM25 main,-,0.8227,0.8244,0.8244,0.8244,0.7012,0.7595,0.6190
3,B2,Bi-encoder MiniLM-L6 title (zero-shot),-,0.8202,0.8281,0.8281,0.8281,0.7091,0.7672,0.6062
4,B2,Bi-encoder MiniLM-L6 main (zero-shot),-,0.8264,0.8306,0.8306,0.8306,0.7137,0.7702,0.6136
5,B2,Bi-encoder bge-base title (zero-shot),-,0.8418,0.8450,0.8450,0.8450,0.7366,0.7981,0.6397
6,B2,Bi-encoder bge-base main (zero-shot),-,0.8429,0.8471,0.8471,0.8471,0.7401,0.8023,0.6433
7,B3,CE MiniLM MS MARCO title (zero-shot),-,0.8330,0.8340,0.8340,0.8340,0.7163,0.7895,0.6248
8,B3,"CE MiniLM title, MSE","42, 43",0.8572,0.8552,0.8549,0.8555,0.7543,0.8175,0.6773
9,D1,"CE MiniLM title_bc, MSE","42, 43",0.8592,0.8584,0.8583,0.8586,0.7594,0.8232,0.6823


## 4. Paired tests

Difference b − a of per-query nDCG on the same queries: mean, 95% paired bootstrap CI, and the Wilcoxon signed-rank p-value (queries with equal nDCG dropped). Val is shown next to test to check that what was seen on val holds on test.

In [6]:
rows = []
for question, a, b in PAIRS:
    r = {'question': question, 'a': NAME[a], 'b': NAME[b]}
    for split in ['val', 'test']:
        pa, pb = per_query[a, split], per_query[b, split]
        p = paired_bootstrap(pa, pb)
        d = (pb.set_index('query_id')['ndcg'] - pa.set_index('query_id')['ndcg']).dropna()
        r[f'{split}_diff'], r[f'{split}_ci95'] = p['diff'], p['ci95']
        if split == 'test':
            r['test_wilcoxon_p'] = float(wilcoxon(d[d != 0]).pvalue) if (d != 0).sum() > 10 else np.nan
            r['test_better'], r['test_worse'] = int((d > 1e-9).sum()), int((d < -1e-9).sum())
    rows.append(r)
paired = pd.DataFrame(rows)
paired['test_wilcoxon_p'] = paired['test_wilcoxon_p'].map(lambda p: f'{p:.1e}')
display(paired)

,question,a,b,val_diff,val_ci95,test_diff,test_ci95,test_wilcoxon_p,test_better,test_worse
0,B2 bge main vs BM25 main,BM25 main,Bi-encoder bge-base main (zero-shot),0.0202,"[0.0151, 0.0253]",0.0227,"[0.0202, 0.0252]",4.0e-77,5108,3657
1,B3 fine-tuning vs zero-shot CE,CE MiniLM MS MARCO title (zero-shot),"CE MiniLM title, MSE (s42)",0.0235,"[0.0197, 0.0277]",0.0209,"[0.0189, 0.0229]",1.6e-131,5537,3167
2,B3 vs BM25 main,BM25 main,"CE MiniLM title, MSE (s42)",0.0338,"[0.0278, 0.0396]",0.0305,"[0.0278, 0.0335]",3.0e-106,5256,3572
3,D1 MiniLM: title_bc vs title (s42),"CE MiniLM title, MSE (s42)","CE MiniLM title_bc, MSE (s42)",0.0023,"[-0.0002, 0.0048]",0.0034,"[0.0023, 0.0046]",1.7e-08,4209,3781
4,D1 MiniLM: main vs title (s42),"CE MiniLM title, MSE (s42)","CE MiniLM main, MSE (s42)",0.0104,"[0.0069, 0.0141]",0.0106,"[0.0088, 0.0122]",1.5e-48,4876,3580
5,D1 DeBERTa: main vs title,"CE DeBERTa title, MSE","CE DeBERTa main, MSE (s42)",0.0077,"[0.0043, 0.011]",0.0102,"[0.0086, 0.0118]",2.0e-48,4752,3466
6,D1 main: DeBERTa vs MiniLM (s42),"CE MiniLM main, MSE (s42)","CE DeBERTa main, MSE (s42)",0.0209,"[0.0163, 0.0253]",0.0220,"[0.0199, 0.0241]",1.4e-135,5456,3117
7,D1 main: DeBERTa vs MiniLM (s43),"CE MiniLM main, MSE (s43)","CE DeBERTa main, MSE (s43)",0.0176,"[0.0129, 0.0222]",0.0196,"[0.0175, 0.0217]",1.2e-112,5361,3191
8,D1 seed: DeBERTa main s43 vs s42,"CE DeBERTa main, MSE (s42)","CE DeBERTa main, MSE (s43)",-0.0001,"[-0.0029, 0.003]",-0.0014,"[-0.0025, -0.0001]",1.2e-03,3763,4058
9,D2 MiniLM: ce4 vs MSE (s42),"CE MiniLM title, MSE (s42)","CE MiniLM title, ce4 (s42)",-0.0006,"[-0.0036, 0.0026]",-0.0005,"[-0.0018, 0.0008]",9.3e-01,4207,4109


## 5. By query type

`source` from the dataset: `other` (most queries), `parse_pattern` (structured queries), `negations` ("without", "not", …), `behavioral`, `nlqec` (natural-language queries, very few). Mean test nDCG per type.

In [7]:
src_of = parts['test'].groupby('query_id')['source'].first()
by_source = pd.DataFrame({NAME[r]: per_query[r, 'test'].set_index('query_id')['ndcg'] for r in BREAKDOWN})
by_source['source'] = src_of.reindex(by_source.index)
by_source = by_source.groupby('source').agg(['mean']).droplevel(1, axis=1).round(4)
by_source.insert(0, 'queries', src_of.value_counts())
display(by_source.sort_values('queries', ascending=False))

,queries,BM25 main,Bi-encoder bge-base main (zero-shot),"CE MiniLM title, MSE (s42)","CE DeBERTa main, MSE (s42)","CE DeBERTa main ce4, mean of 5 fold models","CE DeBERTa-large main ce4, mean of 5 fold models",LightGBM lambdarank on base + large CE + features
source,,,,,,,,
other,7233,0.8283,0.8532,0.8563,0.8915,0.8928,0.9013,0.9031
parse_pattern,781,0.8488,0.8631,0.8726,0.8903,0.8882,0.8946,0.8946
negations,689,0.7510,0.7601,0.8212,0.8448,0.8454,0.8509,0.8508
behavioral,223,0.8516,0.8648,0.8651,0.8870,0.8889,0.8910,0.8928
nlqec,30,0.7225,0.8380,0.7684,0.8526,0.8607,0.8884,0.8750


## 6. By query property

Mean test nDCG by query length, by whether the query has a number, a number with a unit, a color or a negation word, and by whether it names the brand of one of its candidates. The rules are those of the Direction 3 features (`src/features.py`), applied to the query with HTML entities decoded. Groups overlap. `random` is the random order of B0 on the same queries: groups have different label mixes, so a lower nDCG can also mean a harder list. `final - B3` is the paired difference with its 95% bootstrap CI.

In [8]:
import html

import src.features as F

brands = pd.read_parquet(find('products_clean.parquet', PREPROCESS_HINT, INPUT), columns=['product_id', 'brand'])
brands = brands.set_index('product_id')['brand']
qtext = parts['test'].groupby('query_id')['query'].first().map(lambda s: html.unescape(s).lower())
named = [len(b) >= 3 and f' {b} ' in f' {q} ' for b, q in zip(
    parts['test']['product_id'].map(brands).map(lambda b: ' '.join(F.tokens(b))),
    parts['test']['query_id'].map(qtext.map(lambda s: ' '.join(F.tokens(s)))))]
words = qtext.map(lambda s: len(F.tokens(s)))
flags = {'number': qtext.map(lambda s: bool(F.NUM.search(s))),
         'number with unit': qtext.map(lambda s: bool(F.units_of(s))),
         'color': qtext.map(lambda s: bool(F.colors_of(s))),
         'negation word': qtext.map(lambda s: bool(F.NEG.search(s))),
         'brand of a candidate': pd.Series(named).groupby(parts['test']['query_id'].to_numpy()).any().reindex(qtext.index)}
groups = [('length', '1-2 words', words <= 2), ('length', '3 words', words == 3),
          ('length', '4-5 words', words.between(4, 5)), ('length', '6+ words', words >= 6)]
for name, f in flags.items():
    groups += [(name, 'yes', f), (name, 'no', ~f)]

systems = {'random': 'b0_random', **{NAME[r]: r for r in BREAKDOWN}}
pq = {n: per_query[r, 'test'].set_index('query_id') for n, r in systems.items()}
b3 = per_query['b3_minilm_title_mse_s42', 'test']
rows = []
for prop, group, mask in groups:
    ids = mask.index[mask]
    d = paired_bootstrap(b3[b3['query_id'].isin(ids)], per_query[FINAL, 'test'][per_query[FINAL, 'test']['query_id'].isin(ids)])
    rows.append({'property': prop, 'group': group, 'queries': len(ids),
                 **{n: round(float(x.loc[ids, 'ndcg'].mean()), 4) for n, x in pq.items()},
                 'final - B3': d['diff'], 'ci95': d['ci95']})
by_property = pd.DataFrame(rows)
assert by_property.groupby('property')['queries'].sum().eq(len(qtext)).all()
display(by_property)

,property,group,queries,random,BM25 main,Bi-encoder bge-base main (zero-shot),"CE MiniLM title, MSE (s42)","CE DeBERTa main, MSE (s42)","CE DeBERTa main ce4, mean of 5 fold models","CE DeBERTa-large main ce4, mean of 5 fold models",LightGBM lambdarank on base + large CE + features,final - B3,ci95
0,length,1-2 words,1714,0.7576,0.8243,0.8562,0.8526,0.8878,0.8890,0.9005,0.9011,0.0485,"[0.0429, 0.0545]"
1,length,3 words,2342,0.7494,0.8308,0.8524,0.8590,0.8921,0.8912,0.8972,0.9002,0.0412,"[0.0365, 0.0457]"
2,length,4-5 words,3600,0.7475,0.8264,0.8455,0.8561,0.8890,0.8908,0.8987,0.8991,0.0430,"[0.0398, 0.0469]"
3,length,6+ words,1300,0.7196,0.8072,0.8299,0.8474,0.8749,0.8768,0.8836,0.8868,0.0394,"[0.0336, 0.0448]"
4,number,yes,1833,0.7150,0.7942,0.8174,0.8509,0.8823,0.8852,0.8906,0.8917,0.0409,"[0.036, 0.0461]"
5,number,no,7123,0.7538,0.8321,0.8547,0.8560,0.8889,0.8894,0.8980,0.8996,0.0436,"[0.0413, 0.0463]"
6,number with unit,yes,418,0.7296,0.8195,0.8249,0.8668,0.8927,0.8947,0.8982,0.9009,0.0341,"[0.0242, 0.0455]"
7,number with unit,no,8538,0.7467,0.8246,0.8482,0.8543,0.8873,0.8882,0.8964,0.8978,0.0435,"[0.0411, 0.0458]"
8,color,yes,790,0.7515,0.8248,0.8527,0.8529,0.8923,0.8928,0.9006,0.9026,0.0497,"[0.0425, 0.0574]"
9,color,no,8166,0.7453,0.8243,0.8466,0.8551,0.8871,0.8881,0.8961,0.8975,0.0424,"[0.0399, 0.0449]"


## 7. Where each label lands

Mean position of each label inside its list, as a share of the list (0 = top, 1 = bottom; random ≈ 0.5), on test. Exact items should rise and Irrelevant items sink; the gap between E and S is what nDCG rewards most.

In [9]:
def label_positions(run):
    d = parts['test'][['query_id', 'esci_label']].assign(score=score_cache[run, 'test'])
    r = d.groupby('query_id')['score'].rank(ascending=False, method='average')
    n = d.groupby('query_id')['score'].transform('size')
    return ((r - 1) / (n - 1)).groupby(d['esci_label']).mean()


label_ranks = pd.DataFrame({NAME[r]: label_positions(r) for r in BREAKDOWN}).T[['E', 'S', 'C', 'I']].round(3)
label_ranks['S - E'] = (label_ranks['S'] - label_ranks['E']).round(3)
display(label_ranks)

esci_label,E,S,C,I,S - E
BM25 main,0.443,0.542,0.473,0.570,0.099
Bi-encoder bge-base main (zero-shot),0.423,0.543,0.534,0.602,0.120
"CE MiniLM title, MSE (s42)",0.414,0.554,0.552,0.599,0.140
"CE DeBERTa main, MSE (s42)",0.384,0.568,0.629,0.627,0.184
"CE DeBERTa main ce4, mean of 5 fold models",0.383,0.567,0.633,0.632,0.184
"CE DeBERTa-large main ce4, mean of 5 fold models",0.376,0.571,0.638,0.639,0.195
LightGBM lambdarank on base + large CE + features,0.375,0.570,0.635,0.644,0.195


## 8. Per-query wins and losses: final system vs B3

How many test queries the final system improves or hurts compared with the B3 reproduction, and the size of the changes.

In [10]:
qa = per_query['b3_minilm_title_mse_s42', 'test'].set_index('query_id')['ndcg']
qb = per_query[FINAL, 'test'].set_index('query_id')['ndcg']
delta = (qb - qa).dropna().rename('delta').to_frame()
delta['b3'], delta['final'] = qa, qb
info = parts['test'].groupby('query_id').agg(query=('query', 'first'), source=('source', 'first'), n=('esci_label', 'size'),
                                              n_E=('esci_label', lambda l: int((l == 'E').sum())),
                                              n_S=('esci_label', lambda l: int((l == 'S').sum())))
delta = delta.join(info)
bins = pd.cut(delta['delta'], [-1, -0.1, -0.01, -1e-9, 1e-9, 0.01, 0.1, 1],
              labels=['< -0.1', '-0.1..-0.01', '-0.01..0', 'same', '0..0.01', '0.01..0.1', '> 0.1'])
display(bins.value_counts().reindex(bins.cat.categories).rename('queries').to_frame().T)
show = ['query', 'source', 'n', 'n_E', 'n_S', 'b3', 'final', 'delta']
print('Largest gains'); display(delta.sort_values('delta', ascending=False)[show].head(8).round(3))
print('Largest losses'); display(delta.sort_values('delta')[show].head(8).round(3))

,< -0.1,-0.1..-0.01,-0.01..0,same,0..0.01,0.01..0.1,> 0.1
queries,447,1445,548,285,967,3506,1758


Largest gains


,query,source,n,n_E,n_S,b3,final,delta
query_id,,,,,,,,
52737,ifuser,other,16,1,0,0.250,1.000,0.750
75759,onses for men,other,16,1,0,0.256,1.000,0.744
111282,white serger,other,16,1,0,0.292,1.000,0.708
75260,ohio state hoodie 3xl,other,15,1,2,0.281,0.944,0.662
6091,above the ground pool 60,other,16,1,2,0.330,0.989,0.658
88704,rouge marvel select,other,16,2,3,0.353,1.000,0.647
101401,tempered chocolate kit,other,15,1,0,0.356,1.000,0.644
69769,miss c,parse_pattern,13,1,0,0.356,1.000,0.644


Largest losses


,query,source,n,n_E,n_S,b3,final,delta
query_id,,,,,,,,
102156,the looming tower,other,15,1,0,1.000,0.315,-0.685
25265,chelsea stadium soccer cards,other,16,2,1,1.000,0.417,-0.583
28565,contact lens,other,16,1,0,0.996,0.460,-0.536
56301,jordan vogt-roberts,other,16,2,0,0.850,0.334,-0.516
30469,csj,other,16,1,0,1.000,0.500,-0.500
18315,book beach blanket,other,16,4,2,0.930,0.465,-0.465
12755,ball keeper net,other,16,1,11,0.987,0.536,-0.451
60376,laundry room deodorizer,other,16,1,15,1.000,0.550,-0.450


## 9. Worst test queries of the final system

The lowest-nDCG queries that have at least one Exact item, with the full ranked list (label, score, title). Used for the error analysis of the report.

In [11]:
titles = pd.read_parquet(find('products_clean.parquet', PREPROCESS_HINT, INPUT), columns=['product_id', 'title'])
titles = titles.set_index('product_id')['title']
t = parts['test'].assign(score=score_cache[FINAL, 'test'])
pq = per_query[FINAL, 'test'].set_index('query_id')
has_e = t.groupby('query_id')['esci_label'].apply(lambda l: (l == 'E').any())
worst = pq.loc[has_e[has_e].index, 'ndcg'].sort_values().head(6)

rows = []
for qid, nd in worst.items():
    q = t[t['query_id'] == qid].sort_values('score', ascending=False)
    print(f"\n[{q['source'].iloc[0]}] \"{q['query'].iloc[0]}\"  nDCG {nd:.3f}  ({len(q)} candidates)")
    for rank, (_, r) in enumerate(q.iterrows(), 1):
        title = str(titles.get(r['product_id'], ''))[:90]
        if rank <= 8 or r['esci_label'] == 'E':
            print(f"  {rank:2d}. {r['esci_label']}  {r['score']:.3f}  {title}")
        rows.append({'query_id': qid, 'query': r['query'], 'ndcg': round(nd, 4), 'rank': rank,
                     'label': r['esci_label'], 'score': round(float(r['score']), 4), 'title': title})
worst_rows = pd.DataFrame(rows)


[negations] "&#34;tortillas without interesterified soybean oil”"  nDCG 0.191  (39 candidates)
   1. I  0.183  Healthy Harvest Non-GMO Gourmet Soybean Cooking Oil - Healthy Cooking Oil for Cooking, Bak
   2. I  0.176  Soymerica Non-GMO Soybeans - 7 Lbs (Newest Crop). Identity Preserved (IP). Great for Soy M
   3. I  -0.000  BetterBody Foods Avocado Oil, Refined Non-GMO Cooking Oil for Paleo and Keto, 500 Millilit
   4. I  -0.039  Healthy Harvest Canola Oil - Non-GMO Certified with Antioxidants and Omega-3s {One Gallon 
   5. I  -0.211  Laura Soybeans, newest crop and Non-GMO - directly from the Chambers Family Farm in Iowa. 
   6. I  -0.246  Snappy Popcorn 1 Gallon Snappy Pure Peanut Oil No Color Added, 128 Fl Oz
   7. I  -0.257  Amazon Brand - Happy Belly Vegetable Oil, 1 Gallon (128 floz)
   8. I  -0.270  Seapoint Farms Sea Salt Dry Roasted Edamame, 4 oz Gluten-Free Snack (12 pack)
  37. E  -0.908  Nature Made Fish Oil 1000 mg, 250 Softgels Value Size, Fish Oil Omega 3 Supplement Fo


[other] "airport face mask kids"  nDCG 0.279  (15 candidates)
   1. I  0.842  Lewis N. Clark Comfort Eye Mask + Sleep Aid to Block Light for Travel, Airplane, Hotel, Ai
   2. I  -0.185  Seirus Combo Clava Black LG/XL
   3. I  -0.257  Joyoldelf Warmer Balaclava Face Mask Cover Anti-dust Windproof Winter Outdoor Ski Sport
   4. I  -0.465  Warm Fleece Balaclava Ski Bike Full Face Mask Neck Warmer Winter Sports Cap(Black+Red)
   5. I  -0.494  6 Pieces Men Balaclava Full Face Cover Neck Gaiter Ski Scarf UV Sun Protection Bandana Hoo
   6. I  -0.505  Balaclava Neoprene Ski Mask - Cold Weather Face Mask for Men & Women - Windproof Hood Snow
   7. I  -0.539  Balaclava Ski Mask - Winter Baclava - Full Face Motorcycle Black Mask w/Wind, Thermal & UV
   8. I  -0.589  Balaclava Ski Mask - Cold Weather Face Mask for Men & Women - Windproof Hood Snow Gear for
  11. E  -0.801  Sunany Inflatable Neck Pillow Used for Airplanes/Cars/Buses/Trains/Office Napping with Fre


## 10. Supplement: training targets and distillation (val only)

Two questions, each pooled over seeds 42 and 43:

- **ce4 vs MSE on DeBERTa-base main.** D2 compared them on seed 42 only; seed 43 of ce4 comes from notebook 15 (the MSE runs from notebook 05).
- **Distillation vs the MiniLM baselines.** Notebook 14 compares each distilled MiniLM (`kd05`: alpha 0.5, `kd00`: teacher probabilities only) with a ce4 MiniLM trained in the same notebook. For MiniLM, ce4 is a weaker baseline than MSE (D2), so the same students are also compared with **MiniLM main MSE** (`d1_minilm_main_mse`, notebook 05): same model, text, length, learning rate, warm-up and batches; only the training target differs.

How to read the table:
- **Val only.** The supplement runs are ablations of the system, not candidates for it, so their test scores stay blind.
- **Pooled over seeds:** per query, the mean over seeds of (b − a); 95% bootstrap CI over queries (the same computation as in notebook 14, so `kd00 vs ce4` must match `d4_pooled.csv`). The CI covers the sampling of queries, not the training randomness: `per_seed` and `same_sign` show whether both seeds agree, and the `seed noise` rows give the s43 − s42 difference of one configuration for comparison (DeBERTa-base ce4 over five seeds: `deberta_base_seeds.csv` of notebook 15).
- **Recovered share:** (student − MSE) / (teacher − MSE) on the per-query nDCG averaged over seeds, with the teacher = DeBERTa-large CE, mean of 5 fold models (the out-of-fold teacher of notebook 14).

In [ ]:
SUPP_SEEDS = [42, 43]
SUPP = [   # (question, run a, run b), {s} = seed: paired difference b - a on val
    ('D2 DeBERTa-base main: ce4 vs MSE', 'd1_deberta_main_mse_s{s}', 'd2_deberta_main_ce4_s{s}'),
    ('D4 MiniLM main: ce4 vs MSE', 'd1_minilm_main_mse_s{s}', 'd4_minilm_main_ce4_s{s}'),
    ('D4 MiniLM main: distilled (alpha 0.5) vs MSE', 'd1_minilm_main_mse_s{s}', 'd4_minilm_main_kd05_s{s}'),
    ('D4 MiniLM main: distilled (alpha 0) vs MSE', 'd1_minilm_main_mse_s{s}', 'd4_minilm_main_kd00_s{s}'),
    ('D4 MiniLM main: distilled (alpha 0) vs ce4 (as in notebook 14)', 'd4_minilm_main_ce4_s{s}', 'd4_minilm_main_kd00_s{s}'),
]
NOISE = ['d1_deberta_main_mse_s{s}', 'd2_deberta_main_ce4_s{s}',
         'd1_minilm_main_mse_s{s}', 'd4_minilm_main_ce4_s{s}', 'd4_minilm_main_kd00_s{s}']   # s43 - s42


def available(*runs):
    """True if every run has val scores in the inputs (each {s} filled with every seed)."""
    return all(f'{r.format(s=s)}_val.parquet' in SCORES for r in runs for s in SUPP_SEEDS)


def val_ndcg(run):
    """Per-query val nDCG of a run, indexed by query_id (scored once and cached)."""
    if (run, 'val') not in per_query:
        metrics[run, 'val'], per_query[run, 'val'] = summarize(parts['val'], scores(run, 'val'))
    return per_query[run, 'val'].set_index('query_id')['ndcg']


def boot_ci(x, n_boot=1000, seed=42):
    m = x[np.random.default_rng(seed).integers(0, len(x), (n_boot, len(x)))].mean(axis=1)
    return [round(float(np.percentile(m, 2.5)), 4), round(float(np.percentile(m, 97.5)), 4)]


def pooled(a, b):
    """b - a pooled over SUPP_SEEDS: per query, the mean over seeds of (b_s - a_s)."""
    d = pd.concat({s: val_ndcg(b.format(s=s)) - val_ndcg(a.format(s=s)) for s in SUPP_SEEDS}, axis=1, join='inner').dropna()
    x = d.mean(axis=1).to_numpy()
    per_seed = [round(float(d[s].mean()), 4) for s in SUPP_SEEDS]
    return {'diff': round(float(x.mean()), 4), 'ci95': boot_ci(x), 'per_seed': per_seed,
            'same_sign': len({np.sign(v) for v in per_seed}) == 1,
            'wilcoxon_p': f'{wilcoxon(x[x != 0]).pvalue:.1e}', 'queries': len(x)}


def recovered(student, base='d1_minilm_main_mse_s{s}', n_boot=1000, seed=42):
    """(student - base) / (teacher - base) on per-query val nDCG averaged over seeds, with a bootstrap CI."""
    def over_seeds(r):
        return pd.concat([val_ndcg(r.format(s=s)) for s in SUPP_SEEDS], axis=1, join='inner').mean(axis=1)
    d = pd.concat({'s': over_seeds(student), 'r': over_seeds(base), 't': val_ndcg(CE5L)},
                  axis=1, join='inner').dropna().to_numpy()
    share = lambda m: (m[..., 0] - m[..., 1]) / (m[..., 2] - m[..., 1])
    boot = share(d[np.random.default_rng(seed).integers(0, len(d), (n_boot, len(d)))].mean(axis=1))
    return round(float(share(d.mean(axis=0))), 3), [round(float(np.percentile(boot, q)), 3) for q in (2.5, 97.5)]


rows, used = [], set()
for q, a, b in SUPP:
    if available(a, b):
        rows.append({'question': q, 'a': a.format(s='*'), 'b': b.format(s='*'), **pooled(a, b)})
        used |= {r.format(s=s) for r in (a, b) for s in SUPP_SEEDS}
    else:
        print(f'skipped (val scores not in the inputs): {q}')
for run in NOISE:
    if available(run):
        val_ndcg(run.format(s=42)), val_ndcg(run.format(s=43))   # scores them if not done yet
        p = paired_bootstrap(per_query[run.format(s=42), 'val'], per_query[run.format(s=43), 'val'])
        rows.append({'question': 'seed noise: s43 vs s42', 'a': run.format(s=42), 'b': run.format(s=43),
                     'diff': p['diff'], 'ci95': p['ci95']})
supplement = pd.DataFrame(rows) if rows else None
if supplement is not None:
    display(supplement)
if used:
    display(pd.DataFrame([{'run': r, 'val_ndcg': metrics[r, 'val']['ndcg'], 'val_ci95': metrics[r, 'val']['ndcg_ci95']}
                          for r in sorted(used)]))
for v in ['kd05', 'kd00']:
    if available(f'd4_minilm_main_{v}_s{{s}}', 'd1_minilm_main_mse_s{s}'):
        share, ci = recovered(f'd4_minilm_main_{v}_s{{s}}')
        print(f'{v}: recovers {share:.1%} of the val gap between MiniLM main MSE and the teacher '
              f'(95% CI {ci[0]:.1%} to {ci[1]:.1%})')

## 11. Save

In [12]:
OUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT / 'results_all.csv', index=False)
headline.to_csv(OUT / 'results_headline.csv', index=False)
paired.to_csv(OUT / 'paired_tests.csv', index=False)
by_source.to_csv(OUT / 'by_source.csv')
by_property.to_csv(OUT / 'by_query_property.csv', index=False)
label_ranks.to_csv(OUT / 'label_ranks.csv')
delta.sort_values('delta').to_csv(OUT / 'query_deltas.csv')
worst_rows.to_csv(OUT / 'worst_queries.csv', index=False)
if supplement is not None:
    supplement.to_csv(OUT / 'supplement_val.csv', index=False)
print('wrote', sorted(p.name for p in OUT.glob('*.csv')), 'to', OUT)

wrote ['by_query_property.csv', 'by_source.csv', 'label_ranks.csv', 'paired_tests.csv', 'query_deltas.csv', 'results_all.csv', 'results_headline.csv', 'worst_queries.csv'] to D:\Workspace\Web Mining\web-mining-project\data\report
